# Global Military Power Analytics Pip
This project builds an end-to-end analytics workflow starting from web scraping to KPI generation and dashboard-ready dataset creation.

### Import Required Libraries

We import libraries required for:
- Sending HTTP requests
- Parsing HTML content
- Pattern extraction using regex
- Structured dataset creation

In [1]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd


### Extract Country Detail Page Links

We first access the Global Firepower country listing page.
All country-specific detail URLs are extracted dynamically.
Duplicate links are removed to ensure clean iteration.

In [2]:
base_url = "https://www.globalfirepower.com/"
list_url = base_url + "countries-listing.php"

response = requests.get(list_url)
soup = BeautifulSoup(response.text, "html.parser")

country_links = []

for a_tag in soup.find_all("a", href=True):
    href = a_tag["href"]
    if "country-military-strength-detail.php" in href:
        full_link = base_url + href
        country_links.append(full_link)

# Remove duplicates
country_links = list(set(country_links))

print("Total country links found:", len(country_links))
print(country_links[:5])



Total country links found: 145
['https://www.globalfirepower.com//country-military-strength-detail.php?country_id=democratic-republic-of-the-congo', 'https://www.globalfirepower.com//country-military-strength-detail.php?country_id=honduras', 'https://www.globalfirepower.com//country-military-strength-detail.php?country_id=finland', 'https://www.globalfirepower.com//country-military-strength-detail.php?country_id=bulgaria', 'https://www.globalfirepower.com//country-military-strength-detail.php?country_id=nepal']


### Define Country Scraping Function

A reusable function `scrape_country()` is created to:
- Send a request to each country page
- Extract country name
- Extract Power Index rank and score using regex
- Extract all military metrics dynamically

The extracted values are stored in dictionary format.

In [3]:
def scrape_country(url):
    response = requests.get(url, timeout=15)
    soup = BeautifulSoup(response.text, "html.parser")
    
    data = {}
    
    title = soup.find("h1")
    data["country"] = title.get_text(strip=True) if title else None
    
    overview_text = soup.find("span", class_="textNormal")
    if overview_text:
        text = overview_text.get_text(" ", strip=True)
        rank_match = re.search(r"ranked\s+(\d+)", text)
        score_match = re.search(r"score of\s+([\d.]+)", text)
        data["power_index_rank"] = rank_match.group(1) if rank_match else None
        data["power_index_score"] = score_match.group(1) if score_match else None
    
    metric_blocks = soup.find_all("div", class_="specsGenContainers")
    
    for block in metric_blocks:
        label = block.find("span", class_="textLarge")
        value = block.find("span", class_="textWhite")
        
        if label and value:
            key = (
                label.get_text(strip=True)
                .replace(":", "")
                .lower()
                .replace(" ", "_")
            )
            data[key] = value.get_text(strip=True)
    
    return data


In [4]:
print(country_links[0])


https://www.globalfirepower.com//country-military-strength-detail.php?country_id=democratic-republic-of-the-congo


### Scrape All Country Pages

We loop through each country URL and apply the scraping function.
Each country's data is appended into a master list.

Error handling ensures scraping continues even if one page fails.

In [5]:
all_data = []

for i, link in enumerate(country_links):
    try:
        print(f"[{i+1}/{len(country_links)}] Scraping:", link)
        country_data = scrape_country(link)
        all_data.append(country_data)
    except Exception as e:
        print("Failed:", link)
        print("Error:", e)

print("Scraping Complete.")
print("Total Countries Scraped:", len(all_data))


[1/145] Scraping: https://www.globalfirepower.com//country-military-strength-detail.php?country_id=democratic-republic-of-the-congo
[2/145] Scraping: https://www.globalfirepower.com//country-military-strength-detail.php?country_id=honduras
[3/145] Scraping: https://www.globalfirepower.com//country-military-strength-detail.php?country_id=finland
[4/145] Scraping: https://www.globalfirepower.com//country-military-strength-detail.php?country_id=bulgaria
[5/145] Scraping: https://www.globalfirepower.com//country-military-strength-detail.php?country_id=nepal
[6/145] Scraping: https://www.globalfirepower.com//country-military-strength-detail.php?country_id=united-arab-emirates
[7/145] Scraping: https://www.globalfirepower.com//country-military-strength-detail.php?country_id=slovenia
[8/145] Scraping: https://www.globalfirepower.com//country-military-strength-detail.php?country_id=brazil
[9/145] Scraping: https://www.globalfirepower.com//country-military-strength-detail.php?country_id=laos
[1

### Convert Scraped Data into DataFrame

The collected country dictionaries are converted into a structured Pandas DataFrame.
This transforms unstructured web data into tabular format.

In [6]:
df = pd.DataFrame(all_data)

print("Shape of dataset:", df.shape)
df.head()


Shape of dataset: (145, 66)


,country,power_index_rank,power_index_score,purchasing_power_parity,foreign_exchange/gold,defense_budget,external_debt,square_land_area,coastline_coverage,shared_borders,...,coal_deficit,coal_proven_reserves,internet_coverage,labor_force,merchant_marine_fleet,ports_/_trade_terminals,airports,roadway_coverage,railway_coverage,total_tonnage
0,2026Democratic Republic of the Congo Military ...,64,1.3051,80,105,98,41,11,3,125,...,"-304,000 mt",44,48,17,102,44,24,34,47,NaN
1,2026Honduras Military Strength,103,2.2575,106,86,112,40,97,39,35,...,"-144,000 mt",145,35,91,36,39,37,120,97,NaN
2,2026Finland Military Strength,48,0.8673,60,68,40,127,63,46,62,...,"-3,122,000 mt",145,7,105,46,19,48,48,33,50
3,2026Bulgaria Military Strength,62,1.2522,73,51,73,57,99,18,43,...,"+413,000 mt",31,21,101,75,45,43,112,45,NaN
4,2026Nepal Military Strength,124,2.8037,82,76,121,30,90,999,72,...,"-1,082,000 mt",74,37,63,145,145,70,77,120,NaN


### Convert Scraped Data into DataFrame

The collected country dictionaries are converted into a structured Pandas DataFrame.
This transforms unstructured web data into tabular format.

In [7]:
df.to_csv("military_raw_data.csv", index=False)
print("File saved in current notebook folder.")


File saved in current notebook folder.
